# 📓 04 — LeRobot Local InferenceRun HuggingFace VLA models directly — no server needed. Supports ACT, Pi0, SmolVLA,Diffusion policies, and any model LeRobot can load.**Requires**: `pip install "strands-robots[lerobot]"` and a GPU (recommended)

## Architecture```Robot Observation (dict)  → ProcessorBridge.preprocess (normalize, device, crop)  → LeRobot PreTrainedPolicy.select_action / predict_action_chunk  → ProcessorBridge.postprocess (unnormalize, delta-action)  → Action dict (per-joint values)```

## Policy ResolutionThe resolution system finds the correct LeRobot class from HF config.json — handlesLeRobot 0.4, 0.5+ with draccus registry, and custom third-party models.```Resolution strategies (in order):1. PreTrainedConfig draccus resolution (LeRobot 0.5+)2. Manual config.json reading (fallback)3. Direct submodule import: lerobot.policies.{type}.modeling_{type}4. Package-level import: lerobot.policies.{type}5. Legacy factory: lerobot.policies.factory.get_policy_class```

In [ ]:
# Requires: pip install "strands-robots[lerobot]" + GPU# This cell demonstrates the API — uncomment to actually load a modelfrom strands_robots.policies.lerobot_local.resolution import (    resolve_policy_class_by_name,    # resolve_policy_class_from_hub,)# Resolve by explicit typetry:    ACTPolicy = resolve_policy_class_by_name("act")    print(f"ACT policy class: {ACTPolicy}")except ImportError as e:    print(f"LeRobot not installed: {e}")

## Loading a Model```pythonfrom strands_robots import create_policy# Smart string resolution: auto-detects provider from HF model IDpolicy = create_policy("lerobot/act_aloha_sim_transfer_cube_human")# Or explicit:policy = create_policy(    "lerobot_local",    pretrained_name_or_path="lerobot/act_aloha_sim_transfer_cube_human",    device="cuda",    actions_per_step=1,)# Set state keys from your robotpolicy.set_robot_state_keys([    "waist", "shoulder", "elbow", "forearm_roll",    "wrist_angle", "wrist_rotate", "gripper"])# Run inferenceactions = policy.get_actions_sync(observation, "pick up the cube")```

## ProcessorBridgeIntegrates LeRobot's `DataProcessorPipeline` — handles normalization, device transfer,observation formatting, and action unnormalization using the model's own configs.

In [ ]:
from strands_robots.policies.lerobot_local.processor import (    ProcessorBridge, PREPROCESSOR_CONFIG, POSTPROCESSOR_CONFIG)# A passthrough bridge (no pipeline loaded)bridge = ProcessorBridge()print(f"Active: {bridge.is_active}")print(f"Has preprocessor: {bridge.has_preprocessor}")print(f"Has postprocessor: {bridge.has_postprocessor}")print(f"Info: {bridge.get_info()}")# To load from a model:# bridge = ProcessorBridge.from_pretrained("lerobot/act_aloha_sim")# processed_obs = bridge.preprocess(raw_observation, instruction="pick cube")# final_action = bridge.postprocess(raw_action_tensor)

## Real-Time Chunking (RTC)For flow-matching policies (Pi0, SmolVLA), RTC blends action chunks acrossinference calls to compensate for compute latency.```pythonpolicy = create_policy(    "lerobot_local",    pretrained_name_or_path="lerobot/pi0_aloha",    rtc_enabled=True,              # auto-detected from model config if None    rtc_execution_horizon=10,      # timesteps from prefix for guidance    rtc_max_guidance_weight=10.0,  # max correction weight)# RTC is transparent — same get_actions() API# Internally uses predict_action_chunk() with prev_chunk for temporal blending# Tracks inference latency (p95) to estimate delay and skip stale steps```Key RTC internals:- `_predict_with_rtc()`: Calls `predict_action_chunk()` with `prev_chunk_left_over`- `_estimate_inference_delay()`: Uses p95 latency to compute steps consumed during inference- Auto-detects from `model.config.rtc_config.enabled`

## VLA Language TokenizationFor VLA models that need language tokens (xVLA, Pi0 with language), the policyauto-resolves a tokenizer and injects tokens into the observation batch.```python# Resolution order:# 1. config.tokenizer_name (explicit)# 2. config.vlm_model_name (VLM's tokenizer)# 3. policy.processor.tokenizer (built-in)```

## Episode Reset**Important**: Call `policy.reset()` between episodes to clear internal state.```pythonfor episode in range(num_episodes):    policy.reset()  # Clears action queues, RTC state, processor stats    obs = env.reset()    for step in range(max_steps):        actions = policy.get_actions_sync(obs, instruction)        obs = env.step(actions[0])```